In [ ]:
# SCRIPT TO OBTAIN 
import os
import cv2
import mediapipe as mp
import pandas as pd
from tqdm import tqdm  

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.5)

DATA_DIR = 'data/asl_alphabet_train'
data = []

if not os.path.exists(DATA_DIR):
    print(f"Error: {DATA_DIR} not found!")
else:
    for label in os.listdir(DATA_DIR):
        img_folder = os.path.join(DATA_DIR, label)
        if not os.path.isdir(img_folder): continue
        
        print(f"Processing letter: {label}")
        
        for img_name in tqdm(os.listdir(img_folder)): 
            img = cv2.imread(os.path.join(img_folder, img_name))
            
            if img is None: continue
                
            results = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    # Creating the 63-number row
                    row = [lm.x for lm in hand_landmarks.landmark] + \
                          [lm.y for lm in hand_landmarks.landmark] + \
                          [lm.z for lm in hand_landmarks.landmark]
                    row.append(label)
                    data.append(row)

    df = pd.DataFrame(data)

    columns = []
    for i in range(21):
        columns.extend([f'x{i}', f'y{i}', f'z{i}'])
    columns.append('label')

    df.columns = columns

    df.to_csv('asl_landmarks.csv', index=False)
    print("Success! Your landmark dataset is ready.")

hands.close() 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
import torch

df = pd.read_csv('asl_landmarks_modern.csv')

label_encoder = LabelEncoder()

df['label'] = label_encoder.fit_transform(df['label'])
num_classes = len(label_encoder.classes_)

X = df.drop('label',axis=1).values.astype(np.float32)
y = df['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)



In [ ]:
import torch.nn as nn
import torch.optim as optim

class SignLanguageANN(nn.Module):
    def __init__(self,num_classes):
        super(SignLanguageANN,self).__init__()
        self.fc1 = nn.Linear(63,128)
        self.fc2 = nn.Linear(128,64)
        self.fc3 = nn.Linear(64,num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SignLanguageANN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
        

In [ ]:
from tqdm import tqdm
epochs = 100
best_acc = 0.0



for epoch in range(epochs)
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    
    for batch_X, batch_y in progress_bar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(train_loader)
    
    print(f"Epoch {epoch+1}: Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(model.state_dict(), 'best_asl_model.pth')
        np.save('classes.npy', label_encoder.classes_)

print(f"\nTraining Finished! Best Accuracy: {best_acc:.2f}%")